# CS204 — Fashion MNIST
## Классификация изображений одежды с помощью Keras

Ноутбук основан на предоставленном `example.ipynb` и использует Fashion MNIST: 60 000 обучающих и 10 000 тестовых изображений размером 28×28, относящихся к 10 классам.

Порядок работы: загрузка данных → просмотр изображений → нормализация → создание и обучение нейронной сети → графики loss/accuracy → оценка на тестовой выборке.


## 1. Импорт библиотек и воспроизводимость

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Фиксируем генераторы случайных чисел для воспроизводимости
keras.utils.set_random_seed(42)


## 2. Загрузка Fashion MNIST

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()


In [ ]:
print("Train:", x_train.shape, y_train.shape)
print("Test:", x_test.shape, y_test.shape)


В наборе 60 000 обучающих изображений и 10 000 тестовых изображений. Каждое изображение — матрица 28×28 пикселей в оттенках серого.

In [ ]:
labels = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot"
]


In [ ]:
print("Первые 10 меток:", y_train[:10])
print("Первый класс:", labels[y_train[0]])


## 3. Просмотр примеров изображений

In [ ]:
fig, ax = plt.subplots(5, 5, figsize=(12, 10))
for i in range(25):
    axc = ax[i // 5, i % 5]
    axc.imshow(x_train[i], cmap="gray")
    axc.set_title(labels[y_train[i]])
    axc.set_xticks([])
    axc.set_yticks([])
plt.tight_layout()
plt.show()


## 4. Нормализация

Значения пикселей находятся в диапазоне 0–255. Делим их на 255, чтобы перевести входные данные в диапазон 0–1.

In [ ]:
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0


## 5. Создание модели нейронной сети

Архитектура соответствует примеру курса: вход 28×28 → Flatten → Dense(256, ReLU) → Dense(10, softmax).

In [ ]:
input_layer = keras.Input(shape=(28, 28), name="Input")
h = keras.layers.Flatten(name="Flatten")(input_layer)
h = keras.layers.Dense(256, activation="relu", name="Hidden")(h)
output_layer = keras.layers.Dense(10, activation="softmax", name="Output")(h)

model = keras.Model(input_layer, output_layer)


In [ ]:
model.summary()


Количество обучаемых параметров: `(784 × 256 + 256) + (256 × 10 + 10)`.

In [ ]:
(784 * 256 + 256) + (256 * 10 + 10)


In [ ]:
keras.utils.plot_model(model, show_shapes=True, show_layer_names=True)


## 6. Компиляция модели

Так как у нас 10 классов и метки представлены целыми числами, используем `sparse_categorical_crossentropy`.

In [ ]:
model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)


## 7. Обучение модели

Используем размер пакета 64, 20 эпох и 20% обучающих данных для валидации.

In [ ]:
batch_size = 64
epochs = 20

history = model.fit(
    x_train,
    y_train,
    batch_size=batch_size,
    epochs=epochs,
    validation_split=0.2,
    verbose=1
)


## 8. Графики обучения

In [ ]:
def plot_loss_curves(history):
    history_dict = history.history
    loss_values = history_dict["loss"]
    val_loss_values = history_dict["val_loss"]
    epochs_range = range(1, len(loss_values) + 1)

    plt.figure(figsize=(8, 5))
    plt.plot(epochs_range, loss_values, "bo", label="Training loss")
    plt.plot(epochs_range, val_loss_values, "b", label="Validation loss")
    plt.title("Training and validation loss")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()


def plot_acc_curves(history):
    history_dict = history.history
    acc = history_dict["accuracy"]
    val_acc = history_dict["val_accuracy"]
    epochs_range = range(1, len(acc) + 1)

    plt.figure(figsize=(8, 5))
    plt.plot(epochs_range, acc, "bo", label="Training accuracy")
    plt.plot(epochs_range, val_acc, "b", label="Validation accuracy")
    plt.title("Training and validation accuracy")
    plt.xlabel("Epochs")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.show()


In [ ]:
plot_loss_curves(history)


In [ ]:
plot_acc_curves(history)


Посмотрите на графики: если training loss продолжает снижаться, а validation loss перестаёт снижаться или растёт, это может указывать на переобучение.

## 9. Оценка на тестовой выборке

In [ ]:
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=1)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")


## 10. Примеры предсказаний

In [ ]:
probabilities = model.predict(x_test[:10], verbose=0)
predictions = np.argmax(probabilities, axis=1)

fig, ax = plt.subplots(2, 5, figsize=(12, 5))
for i in range(10):
    a = ax[i // 5, i % 5]
    a.imshow(x_test[i], cmap="gray")
    a.set_title(f"Предсказание: {labels[predictions[i]]}\nИстина: {labels[y_test[i]]}")
    a.set_xticks([])
    a.set_yticks([])
plt.tight_layout()
plt.show()


## 11. Дополнительный эксперимент: два скрытых слоя

В примере курса также предлагается попробовать добавить второй скрытый слой. Следующая ячейка создаёт альтернативную модель; её можно обучить отдельно, чтобы сравнить качество с первой моделью.

In [ ]:
input_layer_2 = keras.Input(shape=(28, 28), name="Input")
h2 = keras.layers.Flatten(name="Flatten")(input_layer_2)
h2 = keras.layers.Dense(256, activation="relu", name="Hidden_1")(h2)
h2 = keras.layers.Dense(256, activation="relu", name="Hidden_2")(h2)
output_layer_2 = keras.layers.Dense(10, activation="softmax", name="Output")(h2)

model_2 = keras.Model(input_layer_2, output_layer_2)
model_2.summary()


In [ ]:
model_2.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)


In [ ]:
history_2 = model_2.fit(
    x_train,
    y_train,
    batch_size=64,
    epochs=20,
    validation_split=0.2,
    verbose=1
)


In [ ]:
test_loss_2, test_accuracy_2 = model_2.evaluate(x_test, y_test, verbose=1)
print(f"Two-hidden-layer test loss: {test_loss_2:.4f}")
print(f"Two-hidden-layer test accuracy: {test_accuracy_2:.4f}")


## Итог

Ноутбук загружает Fashion MNIST напрямую через Keras, обучает модель для распознавания 10 категорий одежды, показывает графики обучения и оценивает качество на тестовых данных.

Если задание требует строго использовать только базовую модель из лекции, достаточно выполнить ячейки до раздела **«Оценка на тестовой выборке»**; последний эксперимент с двумя скрытыми слоями является дополнительным.
